In [1]:
!pip install -q streamlit
!pip install -q joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 2.7 MB/s eta 0:00:00


In [8]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os


# 🔹 Set Google Drive path (Modify if necessary)
drive_path = "/content/"

# 🔹 Function to load the trained model and vectorizer
@st.cache_resource
def load_model():
    model_path = os.path.join(drive_path, "disease_model.pkl")
    vectorizer_path = os.path.join(drive_path, "tfidf_vectorizer.pkl")

    if not os.path.exists(model_path) or not os.path.exists(vectorizer_path):
        st.error("🚨 Model or vectorizer not found! Please train and save them first.")
        return None, None

    clf = joblib.load(model_path)
    vectorizer = joblib.load(vectorizer_path)
    return clf, vectorizer

# 🔹 Load Model & Vectorizer
clf, vectorizer_tfidf = load_model()

# 🔹 Load Dataset for Symptom Selection
@st.cache_data
def load_data():
    data_path = "/content/Simulated_data_based_on_the_prevalence_rates_of_symptoms_for_28_disease.csv"
    return pd.read_csv(data_path)

df = load_data()

# ✅ Streamlit App UI
st.title("🔬 Disease Prediction App")
st.write("**Select symptoms and predict the most likely disease.**")

# 🔹 Extract Unique Symptoms
all_symptoms = set()
for symptoms in df["Symptoms"].astype(str).fillna(""):
    all_symptoms.update(symptoms.split(", "))


# 🔹 User Input: Multi-select Symptoms
selected_symptoms = st.multiselect("🩺 Select Symptoms:", sorted(all_symptoms))

# 🔹 Prediction Button
if st.button("🔍 Predict Disease"):
    if not selected_symptoms:
        st.warning("⚠️ Please select at least one symptom.")
    elif clf is None or vectorizer_tfidf is None:
        st.error("🚨 Model is not loaded. Please check the file paths and restart.")
    else:
        # Convert user input to TF-IDF format
        input_text = ", ".join(selected_symptoms)
        input_vectorized = vectorizer_tfidf.transform([input_text])

        # Make Prediction
        prediction = clf.predict(input_vectorized)
        probability = clf.predict_proba(input_vectorized).max()

        # ✅ Display the Prediction
        st.success(f"🩺 **Predicted Disease:** {prediction[0].capitalize()}")
        st.write(f"🔍 **Confidence Score:** {probability:.2f}")

# 🔹 Data Preview
st.write("📊 **Sample Data:**")
st.dataframe(df.head())


Overwriting app.py


In [10]:
!npm install localtunnel
!streamlit run app.py --server.address=localhost &>/content/logs.txt &
!npx localtunnel --port 8501 & curl https://loca.lt/mytunnelpassword


⠙⠹⠸⠼
up to date, audited 23 packages in 832ms
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠼35.237.3.44⠙your url is: https://mean-donuts-cheat.loca.lt
